# CHiRPE Tutorial: Fused String-Input ONNX With Transcript Voting

This notebook is both a runnable demo and a conceptual walkthrough of the fixed-slot fused CHiRPE ONNX model. The key idea is that more of the inference pipeline has been moved into one portable ONNX graph.

In a normal Python inference pipeline, we might tokenize text in Python, call a PyTorch or ONNX classifier, then run transcript-level voting in Python. In this fused pipeline, the serving client sends strings and a mask, and ONNX Runtime executes tokenization, classification, and voting inside the graph.

High-level pipeline:

```text
Raw transcript
-> Python preprocessing / PSYCHS segmentation / optional summarization
-> 15 fixed segment-summary slots
-> ONNX tokenizer
-> ONNX classifier
-> ONNX transcript voting
-> transcript-level prediction
```

Why this matters for deployment: the model boundary is simpler. A client or Triton server request only needs `text[15]` and `segment_mask[15]`; it does not need Python tokenizer code or Python voting code.


## Input Contract

One ONNX request represents one transcript. The model expects exactly 15 segment slots:

```text
text: tensor(string)[15]
segment_mask: tensor(float)[15]
```

`text` contains segment summaries. `segment_mask` tells the graph which slots are real and which slots are padding. A value of `1.0` means active; a value of `0.0` means padded.

The model uses 15 slots because the serving graph is fixed-shape and CHiRPE/PSYCHS operates over a bounded set of segment domains. Fixed shapes are easier to export, validate, and deploy in ONNX Runtime or Triton.

Example for 3 real segments:

```python
text = [
    "segment 1 summary",
    "segment 2 summary",
    "segment 3 summary",
] + [""] * 12

segment_mask = [1.0, 1.0, 1.0] + [0.0] * 12
```

The padded slots still flow through tokenization and classification, so they will have segment-level logits and labels. However, transcript-level voting multiplies by `segment_mask`, so padded slots do not affect the final transcript prediction.


## What Is Inside ONNX?

Inside ONNX:

- BERT tokenizer custom op from ONNX Runtime Extensions
- Transformer classifier
- Segment-level `Softmax` probabilities
- Segment-level `ArgMax` labels
- Masked average transcript voting
- Masked majority transcript voting

Outside ONNX:

- Raw transcript loading
- PSYCHS segmentation
- Optional summarization
- Packing summaries into `text[15]` and `segment_mask[15]`

Shape roadmap for the graph:

```text
text                                  [15]
text[i:i+1] for one tokenizer branch  [1]
tokenized ids after pad/slice         [max_length]
one unsqueezed token row              [1, max_length]
concatenated classifier batch         [15, max_length]
classifier logits                     [15, 2]
segment probabilities                 [15, 2]
segment labels                        [15]
transcript probabilities              [2]
transcript labels                     [1]
```


In [1]:
from pathlib import Path
import json
import sys

import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))

print("Repository root resolved.")

Repository root resolved.


In [2]:
import onnxruntime as ort
from onnxruntime_extensions import get_library_path

from chirpe.data.preprocessor import TranscriptPreprocessor

2026-05-21 11:46:21.613560201 [W:onnxruntime:Default, device_discovery.cc:325 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:92 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


## Choose A Fused Voting Model

This cell selects which exported fused ONNX model to run. The examples default to BERT. Change `BACKBONE` to `clinicalbert` or `mentalbert` if those fixed-slot models have been generated.

What to look for in the output: the printed contract should confirm that the model receives `text` as strings and `segment_mask` as floats.


In [3]:
BACKBONE = "bert"  # bert | clinicalbert | mentalbert
MAX_SEGMENTS = 15

MODEL_NAME = f"chirpe_{BACKBONE}_string_voting"
MODEL_DIR = REPO_ROOT / "outputs/string_onnx_fused_voting" / MODEL_NAME
MODEL_PATH = MODEL_DIR / "1/model.onnx"
METADATA_PATH = MODEL_DIR / "1/export_metadata.json"
CONFIG_PATH = MODEL_DIR / "config.pbtxt"

for path in [MODEL_PATH, METADATA_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Build fixed-slot fused voting models with: \
python scripts/onnx/build_fused_string_voting_onnx.py --max-segments 15"
        )

with open(METADATA_PATH, "r") as file:
    metadata = json.load(file)

print(f"Model: {MODEL_PATH.relative_to(REPO_ROOT)}")
print(f"Metadata: {METADATA_PATH.relative_to(REPO_ROOT)}")
print(json.dumps(metadata["contract"], indent=2))

Model: outputs/string_onnx_fused_voting/chirpe_bert_string_voting/1/model.onnx
Metadata: outputs/string_onnx_fused_voting/chirpe_bert_string_voting/1/export_metadata.json
{
  "text": "tensor(string)[15]",
  "segment_mask": "tensor(float)[15], with 1 for active slots and 0 for padded slots"
}


## Load ONNX Runtime Session

ONNX Runtime executes ONNX graphs. This graph contains a tokenizer node from ONNX Runtime Extensions, not from standard ONNX. That is why we register the ORT Extensions shared library before creating the session.

If the custom-op library is not registered, ONNX Runtime can load ordinary math nodes such as `Softmax`, but it will not know how to execute the tokenizer node.


In [4]:
ortx_library = Path(get_library_path())
if not ortx_library.exists():
    raise FileNotFoundError(f"ORT Extensions library not found: {ortx_library}")

session_options = ort.SessionOptions()
session_options.register_custom_ops_library(str(ortx_library))
session = ort.InferenceSession(str(MODEL_PATH), session_options, providers=["CPUExecutionProvider"])

print("Registered ORT Extensions: OK")

Registered ORT Extensions: OK


In [5]:
print("Inputs:")
for item in session.get_inputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})

print("\nOutputs:")
for item in session.get_outputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})

Inputs:
{'name': 'text', 'type': 'tensor(string)', 'shape': [15]}
{'name': 'segment_mask', 'type': 'tensor(float)', 'shape': [15]}

Outputs:
{'name': 'logits', 'type': 'tensor(float)', 'shape': [15, 2]}
{'name': 'probabilities', 'type': 'tensor(float)', 'shape': [15, 2]}
{'name': 'label', 'type': 'tensor(int64)', 'shape': [15]}
{'name': 'transcript_probabilities', 'type': 'tensor(float)', 'shape': [2]}
{'name': 'transcript_label_average', 'type': 'tensor(int64)', 'shape': [1]}
{'name': 'transcript_label_majority', 'type': 'tensor(int64)', 'shape': [1]}


## Helper Functions

These helpers keep the tutorial readable. `prepare_fixed_slots` converts a variable number of segment summaries into the fixed `text[15]` and `segment_mask[15]` contract. `run_fused_voting` calls ONNX Runtime. The print helpers separate segment-level outputs from transcript-level outputs.


In [6]:
OUTPUT_NAMES = [
    "logits",
    "probabilities",
    "label",
    "transcript_probabilities",
    "transcript_label_average",
    "transcript_label_majority",
]
LABELS = {0: "Healthy", 1: "CHR-P"}


def prepare_fixed_slots(segments, max_segments=15):
    if len(segments) > max_segments:
        print(f"Warning: truncating {len(segments)} segments to {max_segments}.")
    active_segments = segments[:max_segments]
    summaries = [segment["summary"] for segment in active_segments]
    text = summaries + [""] * (max_segments - len(summaries))
    segment_mask = [1.0] * len(summaries) + [0.0] * (max_segments - len(summaries))
    return np.array(text, dtype=object), np.array(segment_mask, dtype=np.float32), active_segments


def run_fused_voting(text, segment_mask):
    outputs = session.run(OUTPUT_NAMES, {"text": text, "segment_mask": segment_mask})
    return dict(zip(OUTPUT_NAMES, outputs))


def print_segment_table(active_segments, result):
    probabilities = result["probabilities"]
    labels = result["label"]
    print("slot | active | domain | label | P(Healthy) | P(CHR-P) | summary preview")
    print("-" * 110)
    for index in range(MAX_SEGMENTS):
        active = index < len(active_segments)
        domain = active_segments[index]["domain"] if active else "<padded>"
        preview = active_segments[index]["summary"][:60].replace("\n", " ") if active else ""
        label = LABELS[int(labels[index])]
        print(
            f"{index:>4} | {str(active):>6} | {domain:<24} | {label:<7} | "
            f"{probabilities[index, 0]:>10.4f} | {probabilities[index, 1]:>8.4f} | {preview}"
        )


def print_transcript_result(result):
    probs = result["transcript_probabilities"]
    avg_label = int(result["transcript_label_average"][0])
    majority_label = int(result["transcript_label_majority"][0])
    print("Transcript probabilities:", {"Healthy": float(probs[0]), "CHR-P": float(probs[1])})
    print("Average-vote label:", avg_label, LABELS[avg_label])
    print("Majority-vote label:", majority_label, LABELS[majority_label])

## Manual Segmented Example

Start with a small pre-segmented transcript. In a real application, these segment summaries would come from the CHiRPE preprocessor or another upstream segmentation/summarization step.

This example is useful because we can see the padding behavior clearly: 3 real summaries become 3 active slots and 12 padded slots. The padded slots may still receive model scores, but the final voting result should ignore them.


In [7]:
manual_segments = [
    {
        "domain": "P1_Unusual_Thoughts",
        "summary": "The participant reports that ordinary events sometimes feel personally meaningful and connected to them.",
    },
    {
        "domain": "P2_Suspiciousness",
        "summary": "The participant describes occasional suspiciousness and worries that others may be watching them.",
    },
    {
        "domain": "P6_Experiences",
        "summary": "The participant mentions unusual perceptual experiences but reports they are intermittent.",
    },
]

text, segment_mask, active_segments = prepare_fixed_slots(manual_segments, max_segments=MAX_SEGMENTS)

print("text shape:", text.shape, text.dtype)
print("segment_mask shape:", segment_mask.shape, segment_mask.dtype)
print("active slots:", int(segment_mask.sum()))
print("first five text slots:")
for item in text[:5]:
    print(repr(item))

text shape: (15,) object
segment_mask shape: (15,) float32
active slots: 3
first five text slots:
'The participant reports that ordinary events sometimes feel personally meaningful and connected to them.'
'The participant describes occasional suspiciousness and worries that others may be watching them.'
'The participant mentions unusual perceptual experiences but reports they are intermittent.'
''
''


In [8]:
manual_result = run_fused_voting(text, segment_mask)

print_segment_table(active_segments, manual_result)
print()
print_transcript_result(manual_result)

slot | active | domain | label | P(Healthy) | P(CHR-P) | summary preview
--------------------------------------------------------------------------------------------------------------
   0 |   True | P1_Unusual_Thoughts      | CHR-P   |     0.2547 |   0.7453 | The participant reports that ordinary events sometimes feel 
   1 |   True | P2_Suspiciousness        | CHR-P   |     0.3433 |   0.6567 | The participant describes occasional suspiciousness and worr
   2 |   True | P6_Experiences           | CHR-P   |     0.2491 |   0.7509 | The participant mentions unusual perceptual experiences but 
   3 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   4 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   5 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   6 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   7 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   8 |  Fals

## Validate ONNX Voting With NumPy

The ONNX graph computes masked transcript voting. We recompute the same logic in NumPy to confirm that padded slots do not affect the final transcript outputs.

Average voting in plain language:

```text
masked_probabilities = probabilities * segment_mask
transcript_probabilities = sum(masked_probabilities) / number_of_active_segments
average_label = argmax(transcript_probabilities)
```

Majority voting in plain language:

```text
active_chr_p_votes = sum(segment_labels * segment_mask)
majority_label = active_chr_p_votes * 2 > number_of_active_segments
```


In [9]:
probabilities = manual_result["probabilities"]
labels = manual_result["label"]

masked_probabilities = probabilities * segment_mask[:, None]
np_transcript_probabilities = masked_probabilities.sum(axis=0) / max(float(segment_mask.sum()), 1e-6)
np_average_label = int(np_transcript_probabilities.argmax())
np_majority_label = int((labels * segment_mask.astype(np.int64)).sum() * 2 > segment_mask.sum())

checks = {
    "transcript_probabilities": np.allclose(
        manual_result["transcript_probabilities"], np_transcript_probabilities, atol=1e-6
    ),
    "average_label": int(manual_result["transcript_label_average"][0]) == np_average_label,
    "majority_label": int(manual_result["transcript_label_majority"][0]) == np_majority_label,
}

print(json.dumps(checks, indent=2))
assert all(checks.values())

{
  "transcript_probabilities": true,
  "average_label": true,
  "majority_label": true
}


## Example With A Synthetic Raw Transcript

The fixed-slot ONNX model does not perform raw transcript segmentation. We still run CHiRPE preprocessing outside ONNX, then pass the resulting segment summaries to the fused model.

This demonstrates the real application boundary: raw transcript processing remains Python-side, while string tokenization, segment classification, and transcript voting are ONNX-side.


In [10]:
DATA_FILE = REPO_ROOT / "data/synthetic/test.json"
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing synthetic test data: {DATA_FILE}")

with open(DATA_FILE, "r") as file:
    test_items = json.load(file)

sample = test_items[0]
print({"participant_id": sample["participant_id"], "label": sample["label"], "turns": len(sample["transcript"])})

{'participant_id': 'HC_0015', 'label': 'Healthy', 'turns': 16}


In [11]:
preprocessor = TranscriptPreprocessor(segmentation_threshold=0.80, use_llm_summarizer=False)
processed = preprocessor.process_transcript(sample["transcript"], participant_id=sample["participant_id"])
synthetic_segments = processed["segments"]

print(f"Extracted {len(synthetic_segments)} segments.")
for index, segment in enumerate(synthetic_segments[:MAX_SEGMENTS], start=1):
    print(f"{index:>2}. {segment['domain']}: {segment['summary'][:90]}")

Extracted 2 segments.
 1. P12_Derealisation: Does tithem sometimes feel unreal to you? Not that The interviewee can recall. Their exper
 2. P4_Disoriented: 


In [12]:
synthetic_text, synthetic_mask, synthetic_active_segments = prepare_fixed_slots(
    synthetic_segments, max_segments=MAX_SEGMENTS
)
synthetic_result = run_fused_voting(synthetic_text, synthetic_mask)

print_segment_table(synthetic_active_segments, synthetic_result)
print()
print_transcript_result(synthetic_result)

slot | active | domain | label | P(Healthy) | P(CHR-P) | summary preview
--------------------------------------------------------------------------------------------------------------
   0 |   True | P12_Derealisation        | Healthy |     0.5013 |   0.4987 | Does tithem sometimes feel unreal to you? Not that The inter
   1 |   True | P4_Disoriented           | CHR-P   |     0.0599 |   0.9401 | 
   2 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   3 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   4 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   5 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   6 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   7 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   8 |  False | <padded>                 | CHR-P   |     0.0599 |   0.9401 | 
   9 |  False | <padded>                 | CHR-P   |  

## Triton Request Shape

The generated `config.pbtxt` uses `max_batch_size: 0`, so the model receives one transcript request at a time. A Triton HTTP request has two inputs: `text` and `segment_mask`.

This is the serving contract you would show to an application team: send 15 strings and 15 mask values, and request transcript-level outputs. The client does not send token IDs.


In [13]:
triton_payload = {
    "inputs": [
        {
            "name": "text",
            "shape": [MAX_SEGMENTS],
            "datatype": "BYTES",
            "data": text.tolist(),
        },
        {
            "name": "segment_mask",
            "shape": [MAX_SEGMENTS],
            "datatype": "FP32",
            "data": segment_mask.tolist(),
        },
    ],
    "outputs": [
        {"name": "transcript_probabilities"},
        {"name": "transcript_label_average"},
        {"name": "transcript_label_majority"},
    ],
}

print(json.dumps(triton_payload, indent=2)[:2500] + "\n...")

{
  "inputs": [
    {
      "name": "text",
      "shape": [
        15
      ],
      "datatype": "BYTES",
      "data": [
        "The participant reports that ordinary events sometimes feel personally meaningful and connected to them.",
        "The participant describes occasional suspiciousness and worries that others may be watching them.",
        "The participant mentions unusual perceptual experiences but reports they are intermittent.",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        ""
      ]
    },
    {
      "name": "segment_mask",
      "shape": [
        15
      ],
      "datatype": "FP32",
      "data": [
        1.0,
        1.0,
        1.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0
      ]
    }
  ],
  "outputs": [
    {
      "name": "transcript_probabilities"
 

## ONNX Graphs In Plain Language

An ONNX model is a computation graph. A graph is a list of operation nodes connected by named tensors. Instead of Python calling functions one by one, ONNX Runtime executes the nodes in the graph.

A tiny graph looks like this:

```text
input tensor -> operation node -> output tensor
```

For this CHiRPE model, the conceptual graph is:

```text
text[15]
-> tokenizer branches
-> input_ids / attention_mask / token_type_ids
-> classifier
-> logits
-> Softmax probabilities and ArgMax labels
-> masked transcript voting using segment_mask
-> final transcript outputs
```

Standard ONNX covers tensor math operations such as `Mul`, `ReduceSum`, `Div`, `Softmax`, and `ArgMax`. The tokenizer is different: it is a custom op from ONNX Runtime Extensions, so we must register that extension library before loading the session.


## ONNX Nodes Used In This Pipeline

An ONNX node is one operation in the graph. Each node receives one or more tensors and produces one or more tensors. The fused CHiRPE graph is built from many nodes, but the important ones for understanding this notebook are listed below.

| Node | Role In This Pipeline |
|---|---|
| `Slice` | Extracts one segment slot, such as `text[0:1]`, from the `text[15]` input tensor. |
| `BertTokenizer` | Custom ONNX Runtime Extensions node that converts a string into token IDs and attention masks. |
| `Pad` | Adds padding tokens so short token sequences reach the fixed `max_length`. |
| `Slice` after tokenization | Truncates token outputs when text is longer than `max_length`. |
| `Unsqueeze` | Adds a new dimension, turning a token vector from `[max_length]` into `[1, max_length]`. |
| `Concat` | Joins the 15 tokenized segment rows into one classifier batch shaped `[15, max_length]`. |
| Transformer classifier nodes | The exported BERT classifier graph. Internally this contains many attention, matrix multiplication, normalization, and activation nodes. |
| `Softmax` | Converts classifier logits into probabilities for Healthy and CHR-P. |
| `ArgMax` | Selects the highest-probability class for each segment. |
| `Mul` | Applies `segment_mask` so padded slots are zeroed out during voting. |
| `ReduceSum` | Adds active segment probabilities or active CHR-P votes. |
| `Max` | Protects against division by zero when computing average probabilities. |
| `Div` | Divides summed probabilities by the number of active segments. |
| `Greater` | Checks whether CHR-P votes are greater than half of active segments. |
| `Cast` | Converts a boolean majority result into an integer label, `0` or `1`. |
| `Reshape` / `Unsqueeze` | Adjusts output shapes to match the exported model contract. |

The main idea is that ONNX does not store Python code. It stores a graph of smaller tensor operations. What looks like one Python function, such as voting across segments, becomes several ONNX nodes connected together.

Mapping from Python concepts to ONNX nodes:

```text
Python concept: text[i:i+1]
ONNX implementation: Slice

Python concept: tokenizer(text)
ONNX implementation: BertTokenizer custom op

Python concept: pad_or_truncate(tokens, max_length)
ONNX implementation: Pad + Slice

Python concept: np.concatenate(rows)
ONNX implementation: Concat

Python concept: probabilities = softmax(logits)
ONNX implementation: Softmax

Python concept: label = argmax(probabilities)
ONNX implementation: ArgMax

Python concept: probabilities * segment_mask
ONNX implementation: Mul

Python concept: sum(masked probabilities)
ONNX implementation: ReduceSum

Python concept: chr_p_votes > active_segments / 2
ONNX implementation: Greater
```

The next code cell inspects the actual exported ONNX file and counts node types. This is useful because the full graph contains many low-level transformer nodes, while the glossary above focuses on the nodes that explain the pipeline structure.


In [14]:
from collections import Counter

import onnx

onnx_model = onnx.load(str(MODEL_PATH))
node_counts = Counter(node.op_type for node in onnx_model.graph.node)

print(f"Total ONNX nodes: {sum(node_counts.values())}")
print("Most common node types:")
for op_type, count in node_counts.most_common(20):
    print(f"{op_type:<20} {count}")

print("\nPipeline-specific node types:")
for op_type in ["BertTokenizer", "Slice", "Pad", "Unsqueeze", "Concat", "Softmax", "ArgMax", "Mul", "ReduceSum", "Max", "Div", "Greater", "Cast"]:
    print(f"{op_type:<20} {node_counts.get(op_type, 0)}")

Total ONNX nodes: 612
Most common node types:
Add                  122
MatMul               96
Slice                61
Unsqueeze            48
Reshape              48
Transpose            48
Pad                  45
Mul                  28
LayerNormalization   25
Div                  25
BertTokenizer        15
Softmax              13
Erf                  12
Concat               5
Gather               4
ReduceSum            4
Cast                 3
Shape                2
Gemm                 2
ArgMax               2

Pipeline-specific node types:
BertTokenizer        15
Slice                61
Pad                  45
Unsqueeze            48
Concat               5
Softmax              13
ArgMax               2
Mul                  28
ReduceSum            4
Max                  1
Div                  25
Greater              1
Cast                 3


## How The Fused Graph Works

The fixed-slot fused voting builder wires several graph pieces together. The most important implementation detail is that ONNX graphs are static: they do not behave like a Python `for` loop that can dynamically iterate over an arbitrary list of segments. Because this model always accepts 15 slots, the builder creates 15 repeated tokenizer branches.

Step-by-step graph construction:

1. Create 15 tokenizer branches, one for each segment slot.
2. For slot `i`, slice `text[i:i+1]` from the input string tensor. This keeps the shape as `[1]`, which the tokenizer expects.
3. Run the tokenizer custom op for that one string. The string becomes numeric token tensors such as `input_ids` and `attention_mask`.
4. Pad and slice each token output to the fixed sequence length, for example 128 tokens. Short text is padded; long text is truncated.
5. Unsqueeze each token vector into one row shaped `[1, max_length]`. This turns one segment into one classifier batch row.
6. Concatenate the 15 rows into classifier inputs shaped `[15, max_length]`. Now the classifier sees the 15 segment slots as a batch of 15 examples.
7. Feed those tensors into the classifier graph. The classifier outputs logits shaped `[15, 2]`, where the two columns are Healthy and CHR-P.
8. Add `Softmax` and `ArgMax` nodes to produce segment probabilities and labels. `Softmax` converts logits to probabilities; `ArgMax` chooses the highest-probability class per segment.
9. Add `Mul`, `ReduceSum`, `Max`, and `Div` nodes to compute masked average transcript probabilities. `Mul` applies the mask, `ReduceSum` adds active segment probabilities, `Max` protects against division by zero, and `Div` computes the average.
10. Add `ReduceSum`, `Greater`, `Cast`, and `Unsqueeze` nodes to compute masked binary majority voting. `ReduceSum` counts active CHR-P votes, `Greater` checks whether CHR-P has a strict majority, `Cast` converts true/false to 1/0, and `Unsqueeze` gives the final output its expected shape.

Diagram:

```text
text[15]
  +--> slice text[0:1]  -> tokenizer -> pad/slice -> unsqueeze -> row 0 --+
  +--> slice text[1:2]  -> tokenizer -> pad/slice -> unsqueeze -> row 1 --+
  +--> ...                                                              +--> concat -> classifier -> logits[15, 2]
  +--> slice text[14:15] -> tokenizer -> pad/slice -> unsqueeze -> row14 --+

logits -> Softmax -> probabilities[15, 2]
logits -> ArgMax  -> segment labels[15]

probabilities + segment_mask -> masked average voting -> transcript_probabilities[2]
segment labels + segment_mask -> masked majority voting -> transcript_label_majority[1]
```

This is why the model can receive strings directly while still returning transcript-level predictions from ONNX.


## Practical Notes

- `max_segments=15` matches the fixed serving contract for this graph.
- If only 2 segments exist, pass 13 empty strings and mask them as `0.0`.
- Padded slots are ignored by transcript voting, but they still add compute cost because they pass through tokenizer and classifier branches.
- This model handles one transcript per request. It does not batch multiple transcripts in one request.
- Raw transcript segmentation and optional summarization still happen outside ONNX.
- The tokenizer is inside ONNX, so the serving client sends strings, not token IDs.
- Segment-level predictions for padded rows may look meaningful, but they should not be interpreted because their mask is `0.0`.


## Summary

```text
We moved more of the CHiRPE inference pipeline into a portable ONNX graph.
The fused model receives 15 text strings and a 15-value segment mask.
Inside ONNX, it tokenizes each string, classifies each segment, computes segment probabilities, and performs transcript-level voting.
The segment mask lets the graph ignore padded slots, so transcripts with fewer than 15 segments still use the same fixed serving contract.
This makes deployment simpler because a client or Triton request sends strings and masks, not token IDs or custom Python voting logic.
```

This tutorial demonstrated how to:

1. Load a fixed-slot fused string-input ONNX model.
2. Register ONNX Runtime Extensions for tokenizer custom ops.
3. Prepare `text[15]` and `segment_mask[15]`.
4. Run tokenizer, classifier, and transcript voting inside ONNX.
5. Validate masked voting against NumPy.
6. Understand how the tokenizer, classifier, and voting graphs were merged.

Main takeaway: this is not just an exported classifier. It is a more complete inference graph that includes string tokenization and transcript-level decision logic.
